In [ ]:
!pip install --upgrade pip
!pip install nnunetv2[all]  # installs nnU-Net v2 and all optional dependencies
!pip install nibabel SimpleITK  # for NIfTI handling


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 32.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:

# 2️⃣ Check where the command was installed
!which nnunetv2_plan_and_preprocess


In [ ]:
import os

# Add ~/.local/bin to PATH
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")
print(os.environ["PATH"])


/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/root/.local/bin


In [ ]:

# ============================================
# Step 1: Mount Google Drive
# ============================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, glob, shutil, json

# ============================================
# Step 2: Define source directories
# ============================================
data_dir = "/content/drive/MyDrive/FYP/ISLES22"
dwi_dir = os.path.join(data_dir, "DWI")
mask_dir = os.path.join(data_dir, "masks")

dwi_files = sorted(glob.glob(os.path.join(dwi_dir, "*.nii*")))
mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.nii*")))

print(f"Found {len(dwi_files)} DWI images and {len(mask_files)} masks.")

# ============================================
# Step 3: Setup nnU-Net raw dataset folder
# ============================================
RAW_ROOT = "/content/nnUNet_raw/nnUNet_raw_data"  # parent of Task folders
DATASET_ID = "011"
DATASET_NAME = "ISLES2022_DWI"

dataset_folder = os.path.join(RAW_ROOT, f"Task{DATASET_ID}_{DATASET_NAME}")
imagesTr = os.path.join(dataset_folder, "imagesTr")
labelsTr = os.path.join(dataset_folder, "labelsTr")

os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

# ============================================
# Step 4: Copy DWI images
# ============================================
for dwi_file in dwi_files:
    patient_id = os.path.basename(dwi_file).replace(".nii", "").replace(".gz","")
    dst = os.path.join(imagesTr, f"{patient_id}_0000.nii.gz")  # 0000 = modality/channel ID
    shutil.copy(dwi_file, dst)

# ============================================
# Step 5: Copy masks and rename labels
# ============================================
for mask_file in mask_files:
    patient_id = os.path.basename(mask_file).replace(".nii", "").replace(".gz","")
    if "_mask" in patient_id:
        patient_id = patient_id.replace("_mask", "")
    dst = os.path.join(labelsTr, f"{patient_id}.nii.gz")  # nnU-Net v2 expects labels without _0000
    shutil.copy(mask_file, dst)

print("Files copied and renamed to nnU-Net v2 format.")

# ============================================
# Step 6: Generate dataset.json for nnU-Net v2
# ============================================
json_dict = {
    "channel_names": {"0": "DWI"},   # modality/channel
    "labels": {"background": 0, "lesion": 1},
    "numTraining": len(dwi_files),
    "file_ending": ".nii.gz"
}

json_file = os.path.join(dataset_folder, "dataset.json")
with open(json_file, "w") as f:
    json.dump(json_dict, f, indent=4)

print(f"Dataset JSON saved at {json_file}")




Mounted at /content/drive
Found 250 DWI images and 250 masks.
Files copied and renamed to nnU-Net v2 format.
Dataset JSON saved at /content/nnUNet_raw/nnUNet_raw_data/Task011_ISLES2022_DWI/dataset.json


In [ ]:
import nnunetv2.paths as nnunet_paths
import os

# Set internal nnU-Net paths
nnunet_paths.nnUNet_raw = "/content/nnUNet_raw/nnUNet_raw_data"
nnunet_paths.nnUNet_preprocessed = "/content/nnUNet_preprocessed"
nnunet_paths.nnUNet_results = "/content/nnUNet_trained_models"

# Also set environment variables (for CLI use)
os.environ["nnUNetv2_raw_data_base"] = nnunet_paths.nnUNet_raw
os.environ["nnUNetv2_preprocessed"] = nnunet_paths.nnUNet_preprocessed
os.environ["nnUNetv2_results_folder"] = nnunet_paths.nnUNet_results

print("nnU-Net v2 internal paths set correctly:")
print("nnUNet_raw:", nnunet_paths.nnUNet_raw)
print("nnUNet_preprocessed:", nnunet_paths.nnUNet_preprocessed)
print("nnUNet_results:", nnunet_paths.nnUNet_results)


nnU-Net v2 internal paths set correctly:
nnUNet_raw: /content/nnUNet_raw/nnUNet_raw_data
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/nnUNet_trained_models


In [ ]:
from nnunetv2.experiment_planning.plan_and_preprocess_api import preprocess

preprocess(
    dataset_ids=[11],
    configurations=('2d','3d_fullres','3d_lowres'),
    num_processes=(8,4,8),
    verbose=True
)


RuntimeError: Could not find a dataset with the ID 11. Make sure the requested dataset ID exists and that nnU-Net knows where raw and preprocessed data are located (see Documentation - Installation). Here are your currently defined folders:
nnUNet_preprocessed=None
nnUNet_results=None
nnUNet_raw=None
If something is not right, adapt your environment variables.

In [ ]:
# ============================================
# Step 1: Mount Google Drive
# ============================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, glob, shutil, json

# ============================================
# Step 2: Define source directories
# ============================================
data_dir = "/content/drive/MyDrive/FYP/ISLES22"
dwi_dir = os.path.join(data_dir, "DWI")
mask_dir = os.path.join(data_dir, "masks")

dwi_files = sorted(glob.glob(os.path.join(dwi_dir, "*.nii*")))
mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.nii*")))

print(f"Found {len(dwi_files)} DWI images and {len(mask_files)} masks.")

# ============================================
# Step 3: Setup nnU-Net raw dataset folder
# ============================================
RAW_ROOT = "/content/nnUNet_raw/nnUNet_raw_data"
DATASET_ID = 11                  # integer ID
DATASET_NAME = "ISLES2022_DWI"  # descriptive name

# Task folder must be zero-padded: Task011_ISLES2022_DWI
dataset_folder = os.path.join(RAW_ROOT, f"Task{DATASET_ID:03d}_{DATASET_NAME}")
imagesTr = os.path.join(dataset_folder, "imagesTr")
labelsTr = os.path.join(dataset_folder, "labelsTr")

os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

# ============================================
# Step 4: Copy DWI images (with _0000 suffix)
# ============================================
for dwi_file in dwi_files:
    patient_id = os.path.basename(dwi_file).replace(".nii", "").replace(".gz", "")
    dst = os.path.join(imagesTr, f"{patient_id}_0000.nii.gz")  # modality index
    shutil.copy(dwi_file, dst)

# ============================================
# Step 5: Copy masks (labelsTr, no _0000)
# ============================================
for mask_file in mask_files:
    patient_id = os.path.basename(mask_file).replace(".nii", "").replace(".gz", "")
    if "_mask" in patient_id:
        patient_id = patient_id.replace("_mask", "")
    dst = os.path.join(labelsTr, f"{patient_id}.nii.gz")
    shutil.copy(mask_file, dst)

print("Files copied and renamed to nnU-Net v2 format.")

# ============================================
# Step 6: Generate dataset.json for nnU-Net v2
# ============================================
json_dict = {
    "channel_names": {"0": "DWI"},
    "labels": {"background": 0, "lesion": 1},
    "numTraining": len(dwi_files),
    "file_ending": ".nii.gz"
}

json_file = os.path.join(dataset_folder, "dataset.json")
with open(json_file, "w") as f:
    json.dump(json_dict, f, indent=4)

print(f"Dataset JSON saved at {json_file}")

# ============================================
# Step 7: Set environment variables BEFORE importing nnunetv2
# ============================================
os.environ["nnUNetv2_raw_data_base"] = RAW_ROOT
os.environ["nnUNetv2_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNetv2_results_folder"] = "/content/nnUNet_trained_models"

print("Environment variables set correctly.")


Mounted at /content/drive
Found 250 DWI images and 250 masks.
Files copied and renamed to nnU-Net v2 format.
Dataset JSON saved at /content/nnUNet_raw/nnUNet_raw_data/Task011_ISLES2022_DWI/dataset.json
Environment variables set correctly.


In [ ]:
# ============================================
# Step 8: Import nnU-Net v2 AFTER env vars
# ============================================
from nnunetv2.experiment_planning.plan_and_preprocess_api import preprocess

# ============================================
# Step 9: Run preprocessing for Task 11
# ============================================
preprocess(
    dataset_ids=[11],                      # integer dataset ID
    configurations=('2d', '3d_fullres', '3d_lowres'),
    num_processes=(8, 4, 8),
    verbose=True
)


RuntimeError: Could not find a dataset with the ID 11. Make sure the requested dataset ID exists and that nnU-Net knows where raw and preprocessed data are located (see Documentation - Installation). Here are your currently defined folders:
nnUNet_preprocessed=None
nnUNet_results=None
nnUNet_raw=None
If something is not right, adapt your environment variables.